In [8]:
import pandas as pd
import joblib

def load_model_and_encoders():
    """Loads the trained model and encoders."""
    model = joblib.load("../model/final_refined_gradient_boosting.pkl")
    one_hot_encoder = joblib.load("../model/one_hot_encoder.pkl")
    label_encoders = joblib.load("../model/label_encoders.pkl")
    return model, one_hot_encoder, label_encoders

def preprocess_data(data, one_hot_encoder, label_encoders):
    """Preprocesses input data before model inference."""
    one_hot_cols = ["MentHlth", "PhysHlth", "DiffWalk"]
    label_enc_cols = ["GenHlth", "Education", "Income"]
    
    # Apply OneHotEncoder
    encoded_data = one_hot_encoder.transform(data[one_hot_cols])
    encoded_cols = one_hot_encoder.get_feature_names_out(one_hot_cols)
    encoded_df = pd.DataFrame(encoded_data, columns=encoded_cols)

    # Apply LabelEncoder
    label_encoded_df = pd.DataFrame()
    for col in label_enc_cols:
        if col in data.columns:
            label_encoded_df[col + "_encoded"] = label_encoders[col].transform(data[col])
        else:
            label_encoded_df[col + "_encoded"] = 0  # Default value

    # Drop original categorical columns
    data.drop(columns=one_hot_cols + label_enc_cols, inplace=True, errors="ignore")

    # Merge transformed data
    data_preprocessed = pd.concat([data, encoded_df, label_encoded_df], axis=1)
    
    # Check for duplicate columns before reindexing
    print("Columns before reindexing:\n", data_preprocessed.columns)
    duplicates = data_preprocessed.columns[data_preprocessed.columns.duplicated()]
    if duplicates.any():
        print("❌ Duplicate columns found:", list(duplicates))
        data_preprocessed = data_preprocessed.loc[:, ~data_preprocessed.columns.duplicated()]
        print("✅ Duplicates removed!")
    
    # Ensure correct column ordering
    data_preprocessed = data_preprocessed.reindex(columns=model.feature_names_in_, fill_value=0)
    
    return data_preprocessed

# Load model and encoders
model, one_hot_encoder, label_encoders = load_model_and_encoders()

# Load new data for inference
new_data = pd.read_csv("../csvs/deployement_data.csv")
print("✅ New data loaded!")

# Preprocess data
new_data_preprocessed = preprocess_data(new_data, one_hot_encoder, label_encoders)
print("✅ Data successfully preprocessed!")

# Make predictions
predictions = model.predict(new_data_preprocessed)

# Save results
results_df = new_data.copy()
results_df["Predicted_Diabetes_012"] = predictions
results_df.to_csv("../csvs/deployment_results.csv", index=False)

print("✅ Predictions saved to deployment_results.csv!")

# Display preview
display(results_df.head())

✅ New data loaded!
Columns before reindexing:
 Index(['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth_encoded',
       'Sex', 'Age', 'MentHlth_1.0', 'MentHlth_2.0', 'MentHlth_3.0',
       'MentHlth_4.0', 'MentHlth_5.0', 'MentHlth_6.0', 'MentHlth_7.0',
       'MentHlth_8.0', 'MentHlth_9.0', 'MentHlth_10.0', 'MentHlth_11.0',
       'MentHlth_12.0', 'MentHlth_13.0', 'MentHlth_14.0', 'MentHlth_15.0',
       'MentHlth_16.0', 'MentHlth_17.0', 'MentHlth_18.0', 'MentHlth_19.0',
       'MentHlth_20.0', 'MentHlth_21.0', 'MentHlth_22.0', 'MentHlth_23.0',
       'MentHlth_24.0', 'MentHlth_25.0', 'MentHlth_26.0', 'MentHlth_27.0',
       'MentHlth_28.0', 'MentHlth_29.0', 'MentHlth_30.0', 'PhysHlth_1.0',
       'PhysHlth_2.0', 'PhysHlth_3.0', 'PhysHlth_4.0', 'PhysHlth_5.0',
       'PhysHlth_6.0', 'PhysHlth_7.0', 'PhysHlth_8.0', 'Phys

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth_encoded,Sex,Age,Predicted_Diabetes_012
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,3.0,1.0,4.0,0.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,3.0,1.0,12.0,0.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,13.0,0.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,3.0,1.0,11.0,0.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,8.0,0.0
